# Pandas

Практические ячейки к слайдам презентации. Заголовки секций совпадают с заголовками слайдов.


In [ ]:
# Практика к уроку — выполняйте ячейки по порядку
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print

np.random.seed(42)


## Индекс — это не «номер строки», а ключ `(example)`


In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml

raw = fetch_openml("titanic", version=1, as_frame=True, parser="auto")
df = raw.frame.copy()
df["Survived"] = df["survived"].astype(int)
df["Pclass"] = df["pclass"].astype(int)
df["Age"] = pd.to_numeric(df["age"], errors="coerce")
df["Fare"] = pd.to_numeric(df["fare"], errors="coerce")
df["Sex"] = df["sex"]
df["Embarked"] = df["embarked"].astype(str)

filtered = df[df["Age"] > 30]
print("Индекс после фильтра (рваный):", filtered.index[:5].tolist())
flat = filtered.reset_index(drop=True)
print("После reset_index:", flat.index[:5].tolist())



## Булева индексация — магия numpy внутри pandas `(example)`


In [ ]:
mask_age = df["Age"] > 30
mask_class = df["Pclass"] == 1
mask = mask_age & mask_class  # скобки обязательны для составных условий
print("Маска (первые 10):", mask.head(10).tolist())
df.loc[mask, ["Sex", "Age", "Pclass", "Survived"]].head()



## Три способа обратиться к данным: `[]`, `.loc`, `.iloc` `(example)`


In [ ]:
sample = df.head(8).copy()
sample.index = sample.index * 10 + 5  # нестандартные метки
print("loc по метке 45:", sample.loc[45, "Age"])
print("iloc по позиции 0:", sample.iloc[0]["Age"])
sample.loc[45, "Fare_note"] = "vip"



## Самая большая боль: SettingWithCopyWarning `(experiment)`


In [ ]:
import warnings
warnings.filterwarnings("default", category=pd.errors.SettingWithCopyWarning)

subset = df[df["Age"] > 30]  # может быть view
# subset["flag"] = 1  # раскомментируйте — увидите Warning

subset_copy = df[df["Age"] > 30].copy()
subset_copy["flag"] = 1

df.loc[df["Age"] > 30, "high_age"] = 1
print("Через .loc на оригинале — без неоднозначности")



## Пропуски: NaN, None, NaT и эволюция типов `(experiment)`


In [ ]:
print(df[["Age", "Fare"]].isnull().sum())
s_int = pd.Series([1, 2, None], dtype="Int64")
print("Nullable Int64:", s_int.dtype, s_int.tolist())



## Базовая аналитика: `value_counts()`, `nunique()`, `corr()` `(example)`


In [ ]:
print(df["Sex"].value_counts())
print("nunique Embarked:", df["Embarked"].nunique())
num = df[["Survived", "Pclass", "Age", "Fare"]].dropna()
print(num.corr().round(2))



## GroupBy: две операции, которые должен знать ML-инженер `(example)`


In [ ]:
agg = df.groupby("Pclass")["Fare"].mean()
print(agg.round(2))
df = df.copy()
df["fare_vs_class"] = df["Fare"] - df.groupby("Pclass")["Fare"].transform("mean")
df[["Pclass", "Fare", "fare_vs_class"]].head()



## MultiIndex: иерархические индексы после groupby `(example)`


In [ ]:
mi = df.groupby(["Pclass", "Sex"]).agg({"Fare": "mean", "Age": "median"})
print("MultiIndex:", mi.index.names)
flat = mi.reset_index()
flat.head()



## Категории: типы кодирования и когда что применять `(example)`


In [ ]:
dummies = pd.get_dummies(df, columns=["Sex", "Embarked"], drop_first=True)
print("OHE столбцы:", [c for c in dummies.columns if c.startswith("Sex") or c.startswith("Embarked")][:4])
cat = df["Sex"].astype("category")
print("Ordinal codes:", cat.cat.codes.head())



## Быстрый Feature Engineering: условия, биннинг и отсечение `(example)`


In [ ]:
import numpy as np

df_fe = df.copy()
df_fe["is_expensive"] = np.where(df_fe["Fare"] > df_fe["Fare"].median(), 1, 0)
df_fe["fare_bin"] = pd.qcut(df_fe["Fare"].fillna(0), q=4, labels=["Q1", "Q2", "Q3", "Q4"])
upper = df_fe["Fare"].quantile(0.99)
df_fe["Fare_clip"] = df_fe["Fare"].clip(upper=upper)
df_fe[["Fare", "Fare_clip", "is_expensive", "fare_bin"]].head()



## Работа с текстом: стринговый аксессор `.str` `(example)`


In [ ]:
names = df["name"].dropna().head(5)
print("Длины:", names.str.len().tolist())
print("Содержит 'Mr':", names.str.contains("Mr").tolist())



## Время как признак: аксессор `.dt` `(example)`


In [ ]:
dates = pd.date_range("2023-01-01", periods=5, freq="D")
ts = pd.DataFrame({"date": dates})
ts["dow"] = ts["date"].dt.dayofweek
ts["hour_sin"] = np.sin(2 * np.pi * ts["date"].dt.hour / 24)
ts



## pandas <-> numpy: мост между экосистемами `(example)`


In [ ]:
X_num = df.select_dtypes(include="number").drop(columns=["Survived"], errors="ignore")
arr = X_num.fillna(0).to_numpy(dtype="float32")
print(arr.shape, arr.dtype)



## Опасности склейки таблиц: merge и concat `(example)`


In [ ]:
ports = pd.DataFrame({"Embarked": ["S", "C", "Q", "S"], "port": ["Soton", "Cher", "Queen", "Soton_dup"]})
merged = pd.merge(df[["Embarked"]].head(6), ports, on="Embarked", how="left")
print(f"Строк после merge с дублями ключа: {len(merged)}")

left = df.iloc[:3].reset_index(drop=True)
right = df.iloc[5:8].reset_index(drop=True)
pd.concat([left[["Age"]], right[["Fare"]]], axis=1)



## merge_asof: асинхронный join для временных рядов `(example)`


In [ ]:
tx = pd.DataFrame({"ts": pd.to_datetime(["2024-01-01 10:05", "2024-01-01 10:20", "2024-01-01 10:40"]), "amt": [100, 200, 150]})
rates = pd.DataFrame({"ts": pd.to_datetime(["2024-01-01 10:00", "2024-01-01 10:15", "2024-01-01 10:30"]), "rate": [1.0, 1.1, 1.2]})
pd.merge_asof(tx.sort_values("ts"), rates.sort_values("ts"), on="ts", direction="backward")



## pandas <-> sklearn: где возникают проблемы `(example)`


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

features = ["Pclass", "Age", "Fare", "Sex"]
X = df[features]
y = df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), ["Pclass", "Age", "Fare"]),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), ["Sex"]),
])
pipe = Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=500, random_state=42))])
pipe.fit(X_train, y_train)
print(f"Test accuracy: {pipe.score(X_test, y_test):.3f}")



## pandas <-> PyTorch: создание Dataset `(example)`


In [ ]:
try:
    import torch
    from torch.utils.data import TensorDataset

    X_np = df[["Pclass", "Age", "Fare"]].fillna(0).to_numpy(dtype="float32")
    y_np = df["Survived"].to_numpy(dtype="float32")
    t_shared = torch.from_numpy(X_np)
    t_copy = torch.tensor(X_np)
    print("from_numpy shares memory:", t_shared.data_ptr() == X_np.__array_interface__["data"][0])
    ds = TensorDataset(t_copy, torch.tensor(y_np))
    print("Dataset size:", len(ds))
except ImportError:
    print("torch не установлен — пропуск демо")



## Data Leakage: где прячется в pandas `(experiment)`


In [ ]:
# НЕПРАВИЛЬНО: статистика по всему датасету до split
# df["Age"].fillna(df["Age"].mean())

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

X = df[["Age", "Fare"]]
y = df["Survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
imp = SimpleImputer(strategy="median")
X_tr_imp = imp.fit_transform(X_tr)
X_te_imp = imp.transform(X_te)  # медиана только с train
print("NaN после imputer (train):", np.isnan(X_tr_imp).any())



## Итеративная обработка: когда файл не влезает в RAM `(example)`


In [ ]:
import io
buf = io.StringIO("\n".join([f"{i},{i*2}" for i in range(1000)]))
buf.seek(0)
chunks = pd.read_csv(buf, names=["a", "b"], chunksize=200)
total = sum(chunk["a"].sum() for chunk in chunks)
print("Сумма a по chunks:", total)



## Производительность: когда pandas «не тянет» `(experiment)`


In [ ]:
import time
s = pd.Series(range(100_000))
t0 = time.perf_counter(); _ = s.apply(lambda x: x * 2); t1 = time.perf_counter()
t2 = time.perf_counter(); _ = s * 2; t3 = time.perf_counter()
print(f"apply: {t1-t0:.3f}s, vectorized: {t3-t2:.3f}s")



## Оптимизация памяти: downcasting типов `(example)`


In [ ]:
col = df["Sex"].astype(str)
print("object KiB:", col.memory_usage(deep=True) / 1024)
cat = col.astype("category")
print("category KiB:", cat.memory_usage(deep=True) / 1024)



## Форматы хранения: Смерть CSV и восстание Parquet `(example)`


In [ ]:
import tempfile
from pathlib import Path

tmp = Path(tempfile.mkdtemp())
pq = tmp / "titanic.parquet"
try:
    df.head(100).to_parquet(pq, index=False)
    loaded = pd.read_parquet(pq)
    print("dtypes сохранены:", loaded.dtypes.head(3).tolist())
except ImportError as e:
    print("Parquet требует pyarrow: pip install pyarrow")
    print(e)



## Гигиена памяти в Jupyter: борьба с OOM `(example)`


In [ ]:
import gc

big = df.copy()
del big
gc.collect()
print("После del + gc.collect()")

